# 15 — Contrast inspection

Loads `14_contrast_table.csv` and ranks images by band contrast Δ (highest → lowest).

Δ = I_max − I_min on the [0, 1]-normalised smoothed multi-slice profile.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT    = Path('/mnt/storage')
TABLE_CSV    = Path('/mnt/storage/projects/Thesis/LuminousBandsInERing/post_pipeline/14_contrast_table.csv')
GEO_CACHE    = Path('/mnt/storage/projects/Thesis/LuminousBandsInERing/post_pipeline/geometry_cache.csv')
PIPELINE_DIR = DATA_ROOT / 'pipeline_output'

ERING_CORE_KM = 238_400   # Enceladus orbital radius ≈ E ring centre

In [ ]:
df = pd.read_csv(TABLE_CSV)
df = df.sort_values('delta', ascending=False).reset_index(drop=True)
df.index += 1   # rank starts at 1

display(df[['opusid', 'I_max', 'I_min', 'delta',
            'n_max_picks', 'n_min_picks', 'n_flanking_mins']].style
    .format({'I_max': '{:.4f}', 'I_min': '{:.4f}', 'delta': '{:.4f}'})
    .background_gradient(subset=['delta'], cmap='RdYlGn')
    .set_caption('Ranked by contrast Δ (highest → lowest)'))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), dpi=110)

colors = ['#d32f2f' if v >= 0 else '#78909c' for v in df['delta']]
ax.bar(range(len(df)), df['delta'], color=colors, width=0.7)

ax.set_xticks(range(len(df)))
ax.set_xticklabels(
    [o.replace('co-iss-', '') for o in df['opusid']],
    rotation=90, fontsize=7,
)
ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)
ax.set_ylabel('Contrast  Δ = I_max − I_min  [normalised]')
ax.set_title('Band contrast Δ ranked highest → lowest')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
import json
import numpy as np

CISSCAL_DIR  = DATA_ROOT / 'cisscal_output'
PROFILE_DIR  = Path('/mnt/storage/projects/Thesis/LuminousBandsInERing/post_pipeline/13_profile_slices')

# Attach COISSfilter from geometry cache so titles can show the filter name
if GEO_CACHE.exists():
    _geo = pd.read_csv(GEO_CACHE, usecols=['opusid', 'COISSfilter'])
    df = df.merge(_geo, on='opusid', how='left')

N_COLS = 7
n      = len(df)
n_rows = (n + N_COLS - 1) // N_COLS

fig, axes = plt.subplots(n_rows, N_COLS,
                         figsize=(N_COLS * 2.6, n_rows * 2.8), dpi=100)
axes = axes.flatten()

for ax in axes[n:]:
    ax.axis('off')

for i, row in df.reset_index().iterrows():
    ax     = axes[i]
    opusid = row['opusid']
    rank   = int(row['index'])

    filt = row.get('COISSfilter', None)
    filt_str = str(filt) if (filt is not None and pd.notna(filt)) else '?'

    # Load residual
    npz_path = CISSCAL_DIR / opusid / 'residual.npz'
    if not npz_path.exists():
        ax.set_title(f"#{rank}\n{opusid[-10:]}\n(missing)", fontsize=6)
        ax.axis('off')
        continue
    r = np.asarray(np.load(npz_path)['residual'], dtype=np.float32)

    # Symmetric colour limits at 99th percentile of absolute values
    fin  = np.isfinite(r)
    vlim = float(np.percentile(np.abs(r[fin]), 99)) if fin.any() else 1.0
    ax.imshow(r, cmap='RdBu_r', origin='upper', vmin=-vlim, vmax=vlim)

    # Overlay picks
    picks_path = PROFILE_DIR / opusid / 'picks.json'
    if picks_path.exists():
        picks = json.loads(picks_path.read_text())
        for p in picks['picks'].get('max', []):
            ax.scatter([p['x_px']], [p['y_px']], c='#d32f2f', s=18,
                       zorder=5, linewidths=0)
        for p in picks['picks'].get('min', []):
            ax.scatter([p['x_px']], [p['y_px']], c='#ff9800', s=18,
                       zorder=5, linewidths=0)

    ax.set_title(
        f"#{rank}  Δ={row['delta']:.10f}\n{opusid.replace('co-iss-', '')}  {filt_str}",
        fontsize=6, pad=3,
    )
    ax.axis('off')

fig.suptitle('CLEAR-subtracted residual — ranked by contrast Δ (highest → lowest)',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## Geometry trends

For each image, fetch observing geometry from the OPUS metadata API and analyse whether band contrast Δ correlates with:
- Phase angle
- Distance to Enceladus (Cassini ↔ Enceladus centre)
- Distance to Saturn (Cassini ↔ Saturn centre, via ring geometry)
- E ring path length (km of the Cassini → Enceladus line of sight that passes through the E ring volume)

In [ ]:
import json
import time
import requests
import numpy as np

df_contrast = pd.read_csv(TABLE_CSV)

# ── Load cached geometry; fetch what's missing ─────────────────────────────────
if GEO_CACHE.exists():
    geo = pd.read_csv(GEO_CACHE)
    missing = set(df_contrast['opusid']) - set(geo['opusid'])
else:
    geo = pd.DataFrame()
    missing = set(df_contrast['opusid'])

if missing:
    print(f'Fetching geometry for {len(missing)} opusids from OPUS…')
    rows = []
    for opusid in sorted(missing):
        try:
            data = requests.get(
                f'https://opus.pds-rings.seti.org/opus/api/metadata/{opusid}.json', timeout=15
            ).json()
            ring = data.get('Ring Geometry Constraints', {})
            enc  = data.get('Enceladus Surface Geometry Constraints', {})
            iss  = data.get('Cassini ISS Constraints', {})
            wl   = data.get('Wavelength Constraints', {})
            rows.append({
                'opusid':             opusid,
                'COISSfilter':        iss.get('COISSfilter'),
                'wavelength1_um':     wl.get('wavelength1'),
                'wavelength2_um':     wl.get('wavelength2'),
                'phase_min':          ring.get('RINGGEOphase1'),
                'phase_max':          ring.get('RINGGEOphase2'),
                'ring_radius_min_km': ring.get('RINGGEOringradius1'),
                'ring_radius_max_km': ring.get('RINGGEOringradius2'),
                'dist_saturn_km':     ring.get('RINGGEOringcenterdistance1'),
                'dist_enceladus_km':  enc.get('SURFACEGEOenceladus_centerdistance1'),
            })
            time.sleep(0.15)
        except Exception as e:
            print(f'  ⚠  {opusid}: {e}')

    new_geo = pd.DataFrame(rows)
    geo = pd.concat([geo, new_geo], ignore_index=True) if not geo.empty else new_geo
    geo.to_csv(GEO_CACHE, index=False)
    print(f'Cached {len(geo)} rows → {GEO_CACHE}')
else:
    print(f'Geometry loaded from cache ({len(geo)} rows)')

geo.head(3)

In [ ]:
# ── Merge contrast + geometry ──────────────────────────────────────────────────
df = df_contrast.merge(geo, on='opusid', how='left')

# Derived geometry columns
df['phase_mid']          = (df['phase_min'] + df['phase_max']) / 2
df['ring_radius_mid_km'] = (df['ring_radius_min_km'] + df['ring_radius_max_km']) / 2
df['wavelength_mid_nm']  = ((df['wavelength1_um'] + df['wavelength2_um']) / 2 * 1000).round().astype('Int64')

# E ring distance — flag near-edge-on geometry (ring radius > 600 000 km is unphysical)
df['dist_ering_km'] = (df['ring_radius_mid_km'] - ERING_CORE_KM).abs()
df.loc[df['ring_radius_mid_km'] > 600_000, 'dist_ering_km'] = float('nan')

df_valid = df.dropna(subset=['delta', 'phase_mid', 'dist_saturn_km', 'dist_enceladus_km'])
print(f'{len(df_valid)} / {len(df)} rows with full geometry')
print(f'E ring distance available: {df_valid["dist_ering_km"].notna().sum()} rows')

wl_groups = sorted(df_valid['wavelength_mid_nm'].dropna().unique())
print('Wavelength groups (nm):', list(wl_groups))
df_valid[['opusid', 'COISSfilter', 'wavelength_mid_nm', 'phase_mid',
          'dist_saturn_km', 'dist_enceladus_km', 'dist_ering_km', 'delta']]

In [ ]:
import re
import numpy as np
import spiceypy as spice

try:
    from numpy import trapezoid as _trapz   # NumPy ≥ 2.0
except ImportError:
    from numpy import trapz as _trapz       # NumPy < 2.0

# ── E ring volume model (cylindrical annulus in IAU_SATURN frame) ──────────────
ERING_R_INNER_KM = 180_000   # inner radial edge from Saturn spin axis
ERING_R_OUTER_KM = 480_000   # outer radial edge (~8 Saturn radii)
ERING_H_HALF_KM  =   2_000   # vertical half-thickness (slightly above max literature value)

CASSINI_ID     = -82
LOS_CACHE      = Path('/mnt/storage/projects/Thesis/LuminousBandsInERing/post_pipeline/los_ering_cache.csv')
KERNEL_ARCHIVE = DATA_ROOT / 'cosp_1000_070930_120502'
KERNELS_DIR    = KERNEL_ARCHIVE / 'cosp_1000' / 'data'


def _load_spice_kernels() -> bool:
    meta_paths = sorted(KERNEL_ARCHIVE.glob('cas_*_070930_120502.tm'))
    if not meta_paths:
        return False
    rel_paths: set[str] = set()
    for mk in meta_paths:
        text = mk.read_text()
        m = re.search(r'\\begindata(.*?)(?:\\begintext|$)', text, re.DOTALL)
        if not m:
            continue
        rel_paths.update(re.findall(r"'\$KERNELS/([^']+)'", m.group(1)))
    if not rel_paths:
        return False
    spice.kclear()
    loaded = 0
    for rel in sorted(rel_paths):
        full = KERNELS_DIR / rel
        if full.exists():
            try:
                spice.furnsh(str(full))
                loaded += 1
            except Exception:
                pass
    return loaded > 0


def _t_exit_outer(pos_c: np.ndarray, d: np.ndarray, r_outer: float) -> float:
    """
    Largest positive t at which the ray P(t) = pos_c + t*d exits the outer
    cylinder (cyl. radius = r_outer).  Falls back to 1.0 if degenerate.
    """
    a = d[0]**2 + d[1]**2
    b = 2.0 * (pos_c[0]*d[0] + pos_c[1]*d[1])
    c = pos_c[0]**2 + pos_c[1]**2 - r_outer**2
    disc = b**2 - 4.0*a*c
    if a < 1e-20 or disc < 0.0:
        return 1.0
    sq    = np.sqrt(disc)
    t_far = max((-b - sq) / (2.0*a), (-b + sq) / (2.0*a))
    return t_far if t_far > 0.0 else 1.0


def _los_length_in_ering(pos_c: np.ndarray, pos_e: np.ndarray,
                          r_inner: float, r_outer: float, h_half: float,
                          n: int = 2000, t_max: float | None = None) -> float:
    """
    Length (km) of the Cassini ray (directed toward Enceladus) inside the
    E ring volume (r_inner ≤ cyl_r ≤ r_outer  AND  |z| ≤ h_half).

    t_max=None  → extend to the far exit of the outer cylinder (full column).
    t_max=1.0   → stop at Enceladus (near-side column only).
    """
    d = pos_e - pos_c                               # t=1 → Enceladus
    if t_max is None:
        t_max = _t_exit_outer(pos_c, d, r_outer)

    t     = np.linspace(0.0, t_max, n + 1)
    pts   = pos_c + t[:, None] * d
    r_cyl = np.hypot(pts[:, 0], pts[:, 1])
    inside = (
        (r_cyl >= r_inner) & (r_cyl <= r_outer) &
        (np.abs(pts[:, 2]) <= h_half)
    ).astype(float)
    return float(_trapz(inside, t)) * float(np.linalg.norm(d))


def _sclk_from_opusid(opusid: str) -> str:
    """co-iss-n1608970715  →  '1/1608970715'"""
    stem = opusid.split('-')[-1]
    return '1/' + stem[1:]


# ── Clear stale cache ─────────────────────────────────────────────────────────
if LOS_CACHE.exists():
    LOS_CACHE.unlink()
    print('Stale LOS cache removed — recomputing.')

# ── Compute both LOS variants ─────────────────────────────────────────────────
kernels_ok = (spice.ktotal('ALL') > 0) or _load_spice_kernels()
if not kernels_ok:
    print('⚠  SPICE kernels not found — LOS columns will be NaN.')

rows_los = []
for opusid in sorted(df_valid['opusid']):
    full_val = float('nan')
    enc_val  = float('nan')
    if kernels_ok:
        try:
            et    = spice.scs2e(CASSINI_ID, _sclk_from_opusid(opusid))
            pos_c, _ = spice.spkpos('CASSINI',   et, 'IAU_SATURN', 'NONE', 'SATURN')
            pos_e, _ = spice.spkpos('ENCELADUS', et, 'IAU_SATURN', 'NONE', 'SATURN')
            pc, pe = np.asarray(pos_c), np.asarray(pos_e)
            full_val = _los_length_in_ering(pc, pe, ERING_R_INNER_KM, ERING_R_OUTER_KM, ERING_H_HALF_KM)
            enc_val  = _los_length_in_ering(pc, pe, ERING_R_INNER_KM, ERING_R_OUTER_KM, ERING_H_HALF_KM,
                                             t_max=1.0)
        except Exception as exc:
            print(f'  ⚠  {opusid}: {exc}')
    rows_los.append({'opusid': opusid, 'los_ering_km': full_val, 'los_to_enc_km': enc_val})

los_df = pd.DataFrame(rows_los)
los_df.to_csv(LOS_CACHE, index=False)
print(f'Computed and cached {len(los_df)} rows → {LOS_CACHE}')

# ── Attach to df_valid ────────────────────────────────────────────────────────
df_valid = df_valid.merge(los_df[['opusid', 'los_ering_km', 'los_to_enc_km']],
                          on='opusid', how='left')
print(f'los_ering_km  (full):  {df_valid["los_ering_km"].notna().sum()} / {len(df_valid)} rows')
print(f'los_to_enc_km (enc):   {df_valid["los_to_enc_km"].notna().sum()} / {len(df_valid)} rows')
df_valid[['opusid', 'los_ering_km', 'los_to_enc_km', 'delta']].head(8)

In [ ]:
"""
Verification figure — LOS path through the E ring.

Left  — top-down view (IAU_SATURN equatorial x-y plane).
Right — r–z cross-section zoomed to the ring region.

Each ray runs from Cassini to the far exit of the outer E ring cylinder.
Orange = inside E ring volume.  Grey = outside.
"""
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

RS_KM  = 60_268.0
N_SEG  = 400

COL_IN  = '#e65100'
COL_OUT = '#9e9e9e'

# ── Ensure SPICE kernels are loaded ──────────────────────────────────────────
if spice.ktotal('ALL') == 0:
    if not _load_spice_kernels():
        raise RuntimeError('SPICE kernels not found — check that mnt-storage is mounted.')
    print(f'Kernels loaded: {spice.ktotal("ALL")}')

# ── Re-derive 3-D positions ───────────────────────────────────────────────────
positions = {}
for opusid in df_valid['opusid']:
    try:
        et    = spice.scs2e(CASSINI_ID, _sclk_from_opusid(opusid))
        pos_c, _ = spice.spkpos('CASSINI',   et, 'IAU_SATURN', 'NONE', 'SATURN')
        pos_e, _ = spice.spkpos('ENCELADUS', et, 'IAU_SATURN', 'NONE', 'SATURN')
        positions[opusid] = (np.asarray(pos_c, dtype=float),
                             np.asarray(pos_e, dtype=float))
    except Exception as exc:
        print(f'  ⚠  {opusid}: {exc}')
print(f'Positions computed for {len(positions)} / {len(df_valid)} images.')

fig, (ax_xy, ax_rz) = plt.subplots(1, 2, figsize=(14, 6.5), dpi=110)

# ── Panel 1: top-down (x-y) ───────────────────────────────────────────────────
r_out_rs = ERING_R_OUTER_KM / RS_KM
r_in_rs  = ERING_R_INNER_KM / RS_KM
ax_xy.add_patch(plt.Circle((0, 0), r_out_rs, fc='#bbdefb', ec='steelblue',
                            lw=0.8, ls='--', alpha=0.5, zorder=2))
ax_xy.add_patch(plt.Circle((0, 0), r_in_rs,  fc='white',   ec='steelblue',
                            lw=0.8, ls='--', alpha=1.0, zorder=3))
ax_xy.add_patch(plt.Circle((0, 0), 238_400 / RS_KM, fc='none',
                            ec='steelblue', lw=1.5, alpha=0.9, zorder=3))
ax_xy.add_patch(plt.Circle((0, 0), 1.0, fc='wheat', ec='grey', lw=1, zorder=5))

all_r_max = 0.0
for opusid, (pos_c, pos_e) in positions.items():
    d     = pos_e - pos_c
    t_max = _t_exit_outer(pos_c, d, ERING_R_OUTER_KM)
    t_arr = np.linspace(0.0, t_max, N_SEG + 1)
    pts   = pos_c + t_arr[:, None] * d
    r_c   = np.hypot(pts[:, 0], pts[:, 1])
    in_v  = (r_c >= ERING_R_INNER_KM) & (r_c <= ERING_R_OUTER_KM) & \
            (np.abs(pts[:, 2]) <= ERING_H_HALF_KM)
    xs, ys = pts[:, 0] / RS_KM, pts[:, 1] / RS_KM
    all_r_max = max(all_r_max, float(np.hypot(pos_c[0], pos_c[1])) / RS_KM)

    for i in range(N_SEG):
        col = COL_IN if in_v[i] else COL_OUT
        ax_xy.plot(xs[i:i+2], ys[i:i+2], color=col,
                   lw=1.8 if in_v[i] else 0.7,
                   alpha=0.75 if in_v[i] else 0.25,
                   solid_capstyle='round',
                   zorder=4 if in_v[i] else 3)

    ax_xy.scatter(*pos_c[:2] / RS_KM, c='#7b1fa2', s=16, zorder=6, linewidths=0)
    ax_xy.scatter(*pos_e[:2] / RS_KM, c='dodgerblue', s=20, zorder=6, linewidths=0)

r_lim = max(all_r_max * 1.08, r_out_rs * 1.2)
ax_xy.set_xlim(-r_lim, r_lim)
ax_xy.set_ylim(-r_lim, r_lim)
ax_xy.set_aspect('equal')
ax_xy.set_xlabel('x  [Saturn radii]', fontsize=9)
ax_xy.set_ylabel('y  [Saturn radii]', fontsize=9)
ax_xy.set_title('Top-down view  (IAU_SATURN x – y)', fontsize=10)
ax_xy.grid(True, alpha=0.15)
ax_xy.legend(handles=[
    mpatches.Patch(fc=COL_IN,    alpha=0.8, label='LOS inside E ring'),
    mpatches.Patch(fc=COL_OUT,   alpha=0.6, label='LOS outside E ring'),
    mpatches.Patch(fc='#bbdefb', alpha=0.6, label='E ring volume'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#7b1fa2',
               markersize=6, label='Cassini'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='dodgerblue',
               markersize=6, label='Enceladus'),
], loc='lower right', fontsize=8)

# ── Panel 2: r–z cross-section (zoomed to ring region) ───────────────────────
ax_rz.add_patch(mpatches.Rectangle(
    (ERING_R_INNER_KM, -ERING_H_HALF_KM),
    ERING_R_OUTER_KM - ERING_R_INNER_KM, 2 * ERING_H_HALF_KM,
    fc='#bbdefb', ec='steelblue', lw=0.8, ls='--', alpha=0.45, zorder=2,
))
ax_rz.axvline(238_400, color='steelblue', lw=1.5, alpha=0.9, zorder=3,
              label='Enceladus orbit')
ax_rz.axhline(0, color='grey', lw=0.5, ls=':', alpha=0.5)

for opusid, (pos_c, pos_e) in positions.items():
    d     = pos_e - pos_c
    t_max = _t_exit_outer(pos_c, d, ERING_R_OUTER_KM)
    t_arr = np.linspace(0.0, t_max, N_SEG + 1)
    pts   = pos_c + t_arr[:, None] * d
    r_c   = np.hypot(pts[:, 0], pts[:, 1])
    in_v  = (r_c >= ERING_R_INNER_KM) & (r_c <= ERING_R_OUTER_KM) & \
            (np.abs(pts[:, 2]) <= ERING_H_HALF_KM)

    for i in range(N_SEG):
        col = COL_IN if in_v[i] else COL_OUT
        ax_rz.plot(r_c[i:i+2], pts[i:i+2, 2], color=col,
                   lw=1.8 if in_v[i] else 0.6,
                   alpha=0.75 if in_v[i] else 0.2,
                   solid_capstyle='round',
                   zorder=4 if in_v[i] else 3)

ax_rz.set_xlim(ERING_R_INNER_KM * 0.80, ERING_R_OUTER_KM * 1.15)
ax_rz.set_ylim(-ERING_H_HALF_KM * 2.5, ERING_H_HALF_KM * 2.5)
ax_rz.set_xlabel('Cylindrical radius  r  [km]', fontsize=9)
ax_rz.set_ylabel('z  [km]  (out of equatorial plane)', fontsize=9)
ax_rz.set_title('Cross-section view  (r – z, zoomed to ring region)', fontsize=10)
ax_rz.grid(True, alpha=0.15)
ax_rz.legend(fontsize=8)
ax_rz.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e3:.0f}k'))
ax_rz.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e3:.0f}k'))

fig.suptitle('LOS path through the E ring — verification\n'
             f'E ring model: r ∈ [{ERING_R_INNER_KM/1e3:.0f}k, {ERING_R_OUTER_KM/1e3:.0f}k] km, '
             f'|z| ≤ {ERING_H_HALF_KM/1e3:.0f}k km   |   ray extends Cassini → far ring edge',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib as mpl

xvars = [
    ('phase_mid',         'Phase angle (°)',                          'Phase angle'),
    ('dist_enceladus_km', 'Cassini – Enceladus distance (km)',        'Distance to Enceladus'),
    ('dist_saturn_km',    'Cassini – Saturn distance (km)',           'Distance to Saturn'),
    ('los_ering_km',      'LOS inside E ring — full column (km)',     'E ring path (full)'),
]

# Physically motivated colours: UV → violet, visible → true colour, IR → dark reds
WL_COLORS = {
    338:  '#8b00ff',   # UV3  — violet
    451:  '#0055ff',   # BL1  — blue
    568:  '#55cc00',   # GRN  — yellow-green
    650:  '#ff2200',   # RED  — red
    752:  '#bb0000',   # IR1  — dark red
    862:  '#770000',   # IR2  — deeper red
    930:  '#440000',   # IR3  — maroon
}

# Only keep wavelength groups with at least 4 images
wl_counts  = df_valid['wavelength_mid_nm'].value_counts()
wl_groups  = sorted(wl for wl in wl_counts.index if wl_counts[wl] >= 4)
n_wl       = len(wl_groups)
n_vars     = len(xvars)

print('Wavelength groups with n ≥ 4:', {int(wl): int(wl_counts[wl]) for wl in wl_groups})

fig, axes = plt.subplots(n_wl, n_vars,
                         figsize=(n_vars * 4, n_wl * 3),
                         dpi=100, squeeze=False)

for row_i, wl in enumerate(wl_groups):
    grp   = df_valid[df_valid['wavelength_mid_nm'] == wl]
    filt  = grp['COISSfilter'].iloc[0] if not grp.empty else '?'
    color = WL_COLORS.get(int(wl), '#888888')

    for col_i, (xcol, xlabel, title) in enumerate(xvars):
        ax  = axes[row_i, col_i]
        sub = grp.dropna(subset=[xcol, 'delta'])
        ax.scatter(sub[xcol], sub['delta'], c=[color], s=60, alpha=0.85,
                   edgecolors='none', zorder=4)

        x = sub[xcol].to_numpy(dtype=float)
        y = sub['delta'].to_numpy(dtype=float)
        finite = np.isfinite(x) & np.isfinite(y)
        if finite.sum() >= 2 and np.nanmax(x[finite]) > np.nanmin(x[finite]):
            coeffs = np.polyfit(x[finite], y[finite], deg=1)
            x_fit = np.linspace(np.nanmin(x[finite]), np.nanmax(x[finite]), 100)
            y_fit = np.polyval(coeffs, x_fit)
            ax.plot(x_fit, y_fit, color='black', lw=1.4, ls='--', alpha=0.85, zorder=5)

        if row_i == 0:
            ax.set_title(title, fontsize=10, pad=6)
        if col_i == 0:
            ax.set_ylabel(f'{filt}  ({wl} nm)\nΔ', fontsize=8)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2)
        ax.set_ylim(0, df_valid['delta'].max() * 1.05)

fig.suptitle('Band contrast Δ vs observing geometry — wavelengths with n ≥ 4', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()